# EEG Left vs Right Benchmark: Testing Directional Intent from OpenBCI Signals

## Section 1 — Introduction

Left/right EEG classification is substantially harder than jaw-click detection. Jaw activity tends to produce a stronger, more repeatable biosignal, while directional EEG intent has weaker signal-to-noise ratio and more session-to-session variability.

This notebook evaluates **LEFT vs RIGHT** using two complementary approaches:

- a **windowed block-based benchmark** inside labeled LEFT/RIGHT intervals
- an **event-based benchmark** centered around discrete LEFT/RIGHT movement starts

The goal is not to exaggerate performance. It is to document the modeling path, show where the signal is weak, and explain why the final project uses a hybrid design rather than relying on EEG alone.


In [ ]:
import io
import json
import pickle
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, welch
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from google.colab import files
from IPython.display import Markdown, display

plt.rcParams.update(
    {
        "figure.figsize": (14, 6),
        "figure.dpi": 120,
        "axes.facecolor": "#fbfbfd",
        "figure.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.frameon": False,
    }
)

DEFAULT_FS_FALLBACK = 250.0
CLASS_ORDER = ["LEFT", "RIGHT"]
SIGNAL_NAME_RE = re.compile(r"^(ch(?:annel)?|eeg|emg|exg|adc)[ _-]*\d+$", re.IGNORECASE)
INDEX_HINTS = ("sample", "index")
TIME_HINTS = ("time", "timestamp", "ts")
MARKER_HINTS = ("marker", "event", "trigger", "label", "stim", "class")
RESULT_HINTS = ("summary", "result", "metric", "report", "confusion", "coverage")
MODEL_HINTS = (".pkl", ".pickle", ".json")


## Section 2 — Upload Files

Upload one or more files directly into Colab. The notebook will separate likely OpenBCI recordings from optional artifacts such as saved models or result summaries.

If `ipywidgets` is available, a multi-select widget will be shown. A plain Python input fallback is also provided so the notebook still works in simpler Colab sessions.


In [ ]:
print("Upload OpenBCI recordings plus any optional supporting artifacts.")
uploaded_raw = files.upload()

if not uploaded_raw:
    raise RuntimeError("No files were uploaded. Re-run this cell and select one or more files.")

UPLOADED_FILES = {name: blob for name, blob in uploaded_raw.items()}
UPLOADED_FILENAMES = sorted(UPLOADED_FILES.keys())

CSV_EXTENSIONS = (".csv", ".tsv", ".txt")
MARKDOWN_EXTENSIONS = (".md",)


def filename_has_result_hint(name):
    upper_name = name.lower()
    return any(hint in upper_name for hint in RESULT_HINTS)


RECORDING_CSV_FILES = [
    name for name in UPLOADED_FILENAMES
    if name.lower().endswith(CSV_EXTENSIONS) and not filename_has_result_hint(name)
]
RESULT_FILES = [
    name for name in UPLOADED_FILENAMES
    if filename_has_result_hint(name) or name.lower().endswith(MARKDOWN_EXTENSIONS)
]
MODEL_ARTIFACT_FILES = [name for name in UPLOADED_FILENAMES if name.lower().endswith(MODEL_HINTS)]
OTHER_FILES = [
    name for name in UPLOADED_FILENAMES
    if name not in RECORDING_CSV_FILES and name not in RESULT_FILES and name not in MODEL_ARTIFACT_FILES
]

upload_table = pd.DataFrame(
    {
        "Index": np.arange(1, len(UPLOADED_FILENAMES) + 1),
        "Filename": UPLOADED_FILENAMES,
        "Type": [
            "Recording CSV" if name in RECORDING_CSV_FILES else
            "Result/summary" if name in RESULT_FILES else
            "Model artifact" if name in MODEL_ARTIFACT_FILES else
            "Other"
            for name in UPLOADED_FILENAMES
        ],
        "Size (KB)": [round(len(UPLOADED_FILES[name]) / 1024.0, 1) for name in UPLOADED_FILENAMES],
    }
)
display(upload_table)

if not RECORDING_CSV_FILES:
    raise RuntimeError("No recording CSV files were detected. Please upload at least one OpenBCI recording.")

print("Detected recording CSV files:")
display(pd.DataFrame({"Index": np.arange(1, len(RECORDING_CSV_FILES) + 1), "Filename": RECORDING_CSV_FILES}))

if RESULT_FILES:
    print("Optional uploaded result files:")
    display(pd.DataFrame({"Filename": sorted(set(RESULT_FILES))}))

if MODEL_ARTIFACT_FILES:
    print("Optional uploaded model artifacts:")
    display(pd.DataFrame({"Filename": sorted(set(MODEL_ARTIFACT_FILES))}))

if OTHER_FILES:
    print("Other uploaded files (not used automatically):")
    display(pd.DataFrame({"Filename": OTHER_FILES}))

WIDGETS_AVAILABLE = False
CSV_WIDGET = None
try:
    import ipywidgets as widgets
    CSV_WIDGET = widgets.SelectMultiple(
        options=RECORDING_CSV_FILES,
        value=tuple(RECORDING_CSV_FILES),
        description="Files",
        rows=min(10, len(RECORDING_CSV_FILES)),
        layout=widgets.Layout(width="90%"),
    )
    display(CSV_WIDGET)
    WIDGETS_AVAILABLE = True
    print("Widget note: if you change the selection above, rerun the next line or rerun this cell.")
except Exception as exc:
    print(f"ipywidgets not available; using text input fallback only ({exc}).")


def parse_file_selection(choice_text, available_files):
    if not choice_text.strip():
        return list(available_files)
    selected = []
    pieces = [piece.strip() for piece in choice_text.split(",") if piece.strip()]
    for piece in pieces:
        if piece.isdigit():
            idx = int(piece) - 1
            if idx < 0 or idx >= len(available_files):
                raise IndexError(f"Selection {piece} is outside the recording file list.")
            selected.append(available_files[idx])
        else:
            if piece not in available_files:
                raise KeyError(f"'{piece}' was not found among the detected recording files.")
            selected.append(piece)
    deduped = []
    for item in selected:
        if item not in deduped:
            deduped.append(item)
    return deduped


selection_text = input("Enter recording file indices or exact filenames to analyze [default: all]: ")
if WIDGETS_AVAILABLE and CSV_WIDGET is not None and len(CSV_WIDGET.value) > 0 and not selection_text.strip():
    SELECTED_RECORDING_FILES = list(CSV_WIDGET.value)
else:
    SELECTED_RECORDING_FILES = parse_file_selection(selection_text, RECORDING_CSV_FILES)

print("Selected recordings:")
display(pd.DataFrame({"Filename": SELECTED_RECORDING_FILES}))


## Section 3 — Robust OpenBCI CSV Parsing

OpenBCI recordings are not always uniform across sessions. The parser below tries to tolerate:

- header rows or no header rows
- named channels such as `ch1` through `ch8`
- unnamed numeric columns
- extra timestamp or sample-index columns
- marker columns at the end
- mixed EEG/EMG/hybrid files

Sampling rate is estimated when possible and otherwise defaults to **250 Hz**.


In [ ]:
def clean_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def infer_protocol_tag(filename):
    upper_name = filename.upper()
    if "LRJ" in upper_name or "HYBRID" in upper_name:
        return "LRJ"
    if "LR" in upper_name or "LEFT" in upper_name or "RIGHT" in upper_name:
        return "LR"
    return "OTHER"


def guess_delimiter(lines, sample_count=25):
    candidates = [",", "	", ";"]
    scores = {}
    sample = lines[:sample_count]
    for delimiter in candidates:
        scores[delimiter] = sum(line.count(delimiter) for line in sample)
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else ","


def looks_like_header(tokens):
    joined = " ".join(token.strip() for token in tokens)
    return any(char.isalpha() for char in joined)


def find_data_start(lines, delimiter):
    for idx, line in enumerate(lines[:80]):
        tokens = [token.strip() for token in line.split(delimiter)]
        if len(tokens) < 4:
            continue
        numeric_like = 0
        alpha_like = 0
        for token in tokens:
            if token == "":
                continue
            try:
                float(token)
                numeric_like += 1
            except ValueError:
                if any(char.isalpha() for char in token):
                    alpha_like += 1
        if numeric_like >= max(4, len(tokens) // 2):
            return idx
        if alpha_like >= 2 and idx + 1 < len(lines):
            next_tokens = [token.strip() for token in lines[idx + 1].split(delimiter)]
            next_numeric_like = 0
            for token in next_tokens:
                try:
                    float(token)
                    next_numeric_like += 1
                except ValueError:
                    continue
            if next_numeric_like >= max(4, len(next_tokens) // 2):
                return idx + 1
    return 0


def make_unique_columns(columns):
    counts = Counter()
    unique_columns = []
    for idx, name in enumerate(columns):
        candidate = str(name).strip()
        if candidate == "" or candidate.lower().startswith("unnamed"):
            candidate = f"col_{idx}"
        if candidate in counts:
            counts[candidate] += 1
            candidate = f"{candidate}_{counts[candidate]}"
        else:
            counts[candidate] = 0
        unique_columns.append(candidate)
    return unique_columns


def normalize_column_name(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")


def estimate_fs_from_time_series(time_series):
    numeric = clean_numeric(time_series)
    finite = numeric.dropna().to_numpy(dtype=float)
    if len(finite) < 8:
        return None, None, None

    diffs = np.diff(finite)
    diffs = diffs[np.isfinite(diffs) & (diffs > 0)]
    if len(diffs) < 5:
        return None, None, None

    median_dt = float(np.median(diffs))
    if not np.isfinite(median_dt) or median_dt <= 0:
        return None, None, None

    fs_seconds = 1.0 / median_dt
    fs_milliseconds = 1000.0 / median_dt

    valid_index = np.flatnonzero(numeric.notna().to_numpy())
    interpolated = np.interp(np.arange(len(numeric)), valid_index, finite)

    if 20 <= fs_seconds <= 5000:
        time_axis_sec = interpolated - interpolated[0]
        return float(fs_seconds), time_axis_sec, "time column interpreted as seconds"

    if 20 <= fs_milliseconds <= 5000:
        time_axis_sec = (interpolated - interpolated[0]) / 1000.0
        return float(fs_milliseconds), time_axis_sec, "time column interpreted as milliseconds"

    return None, None, None


def detect_columns(df):
    normalized = {column: normalize_column_name(column) for column in df.columns}

    numeric_columns = []
    for column in df.columns:
        numeric_series = clean_numeric(df[column])
        if numeric_series.notna().mean() >= 0.80:
            numeric_columns.append(column)

    index_candidates = []
    time_candidates = []
    for column in df.columns:
        column_name = normalized[column]
        if any(hint in column_name for hint in INDEX_HINTS):
            index_candidates.append(column)
        if any(hint in column_name for hint in TIME_HINTS):
            time_candidates.append(column)

    for column in numeric_columns:
        numeric_series = clean_numeric(df[column]).dropna()
        if len(numeric_series) < 10:
            continue
        values = numeric_series.to_numpy(dtype=float)
        diffs = np.diff(values)
        if len(diffs) == 0:
            continue
        positive_ratio = float(np.mean(diffs > 0))
        if positive_ratio < 0.90:
            continue
        median_dt = float(np.median(np.abs(diffs)))
        if not np.isfinite(median_dt) or median_dt <= 0:
            continue
        fs_seconds = 1.0 / median_dt
        fs_milliseconds = 1000.0 / median_dt
        if 20 <= fs_seconds <= 5000 or 20 <= fs_milliseconds <= 5000:
            if column not in time_candidates:
                time_candidates.append(column)
        elif column not in index_candidates:
            index_candidates.append(column)

    marker_column = None
    marker_name_candidates = [
        column for column in df.columns if any(hint in normalized[column] for hint in MARKER_HINTS)
    ]
    if marker_name_candidates:
        marker_column = marker_name_candidates[0]
    else:
        low_cardinality_candidates = []
        for column in reversed(list(df.columns)):
            if column in index_candidates or column in time_candidates:
                continue
            text_series = df[column].dropna().astype(str).str.strip()
            if text_series.empty:
                continue
            unique_count = text_series.nunique()
            if unique_count <= 20:
                low_cardinality_candidates.append((column, unique_count))
        if low_cardinality_candidates:
            marker_column = low_cardinality_candidates[0][0]

    signal_candidates = [column for column in df.columns if SIGNAL_NAME_RE.match(normalized[column])]
    if not signal_candidates:
        excluded = set(index_candidates + time_candidates + ([marker_column] if marker_column else []))
        ranked_numeric = []
        for column in df.columns:
            if column in excluded:
                continue
            numeric_series = clean_numeric(df[column])
            valid_fraction = float(numeric_series.notna().mean())
            unique_count = int(numeric_series.dropna().nunique())
            if valid_fraction >= 0.80 and unique_count >= 25:
                ranked_numeric.append((column, valid_fraction, unique_count))
        if not ranked_numeric:
            for column in df.columns:
                if column in excluded:
                    continue
                numeric_series = clean_numeric(df[column])
                ranked_numeric.append((column, float(numeric_series.notna().mean()), int(numeric_series.dropna().nunique())))
        signal_candidates = [column for column, _, _ in ranked_numeric[:8]]

    excluded = set(index_candidates + time_candidates + ([marker_column] if marker_column else []))
    signal_candidates = [column for column in signal_candidates if column not in excluded][:8]

    return {
        "index_candidates": index_candidates,
        "time_candidates": time_candidates,
        "signal_candidates": signal_candidates,
        "marker_column": marker_column,
    }


def load_openbci_csv(file_bytes, filename):
    text = file_bytes.decode("utf-8", errors="ignore")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [line for line in text.split("\n") if line.strip()]
    if not lines:
        raise ValueError(f"{filename} appears to be empty.")

    delimiter = guess_delimiter(lines)
    data_start_idx = find_data_start(lines, delimiter)
    data_tokens = [token.strip() for token in lines[data_start_idx].split(delimiter)]

    header_row_idx = None
    if data_start_idx > 0:
        previous_tokens = [token.strip() for token in lines[data_start_idx - 1].split(delimiter)]
        if len(previous_tokens) == len(data_tokens) and looks_like_header(previous_tokens):
            header_row_idx = data_start_idx - 1
    if header_row_idx is None and looks_like_header(data_tokens):
        header_row_idx = data_start_idx

    parse_start_idx = header_row_idx if header_row_idx is not None else data_start_idx
    parse_text = "\n".join(lines[parse_start_idx:])
    read_kwargs = {"sep": delimiter, "engine": "python"}
    if header_row_idx is None:
        read_kwargs["header"] = None

    df = pd.read_csv(io.StringIO(parse_text), **read_kwargs)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
    if df.empty:
        raise ValueError(f"{filename} did not produce a usable table after parsing.")

    if header_row_idx is None:
        df.columns = make_unique_columns([f"col_{idx}" for idx in range(df.shape[1])])
    else:
        df.columns = make_unique_columns(df.columns.tolist())

    column_info = detect_columns(df)
    fs_estimate_hz = DEFAULT_FS_FALLBACK
    fs_source = f"fallback default ({DEFAULT_FS_FALLBACK:.0f} Hz)"
    time_axis_sec = np.arange(len(df), dtype=float) / fs_estimate_hz

    if column_info["time_candidates"]:
        primary_time_column = column_info["time_candidates"][0]
        inferred_fs, inferred_time_axis_sec, inferred_source = estimate_fs_from_time_series(df[primary_time_column])
        if inferred_fs is not None:
            fs_estimate_hz = inferred_fs
            time_axis_sec = inferred_time_axis_sec
            fs_source = inferred_source

    raw_signal_columns = column_info["signal_candidates"]
    signal_df = df.loc[:, raw_signal_columns].apply(clean_numeric).copy()
    standard_signal_columns = [f"Channel_{idx + 1}" for idx in range(signal_df.shape[1])]
    signal_df.columns = standard_signal_columns
    channel_map = dict(zip(standard_signal_columns, raw_signal_columns))

    marker_column = column_info["marker_column"]
    unique_markers = []
    if marker_column is not None:
        marker_series = df[marker_column].dropna().astype(str).str.strip()
        marker_series = marker_series[marker_series != ""]
        unique_markers = marker_series.unique().tolist()[:20]

    return {
        "filename": filename,
        "protocol_tag": infer_protocol_tag(filename),
        "delimiter": delimiter,
        "header_detected": header_row_idx is not None,
        "df": df,
        "signal_df": signal_df,
        "signal_columns": list(signal_df.columns),
        "raw_signal_columns": raw_signal_columns,
        "channel_map": channel_map,
        "marker_column": marker_column,
        "time_column": column_info["time_candidates"][0] if column_info["time_candidates"] else None,
        "fs_estimate_hz": float(fs_estimate_hz),
        "fs_source": fs_source,
        "time_axis_sec": np.asarray(time_axis_sec, dtype=float),
        "duration_sec": float(time_axis_sec[-1]) if len(time_axis_sec) else 0.0,
        "unique_markers": unique_markers,
    }


SESSIONS = {}
FAILED_CSVS = {}
for filename in SELECTED_RECORDING_FILES:
    try:
        SESSIONS[filename] = load_openbci_csv(UPLOADED_FILES[filename], filename)
    except Exception as exc:
        FAILED_CSVS[filename] = str(exc)

if not SESSIONS:
    raise RuntimeError("None of the selected recordings could be parsed successfully.")

if FAILED_CSVS:
    print("Some files could not be parsed automatically:")
    display(pd.DataFrame([{"Filename": name, "Error": err} for name, err in FAILED_CSVS.items()]))

summary_rows = []
for filename, session in SESSIONS.items():
    signal_text = ", ".join([f"{std}={raw}" for std, raw in session["channel_map"].items()])
    summary_rows.append(
        {
            "Filename": filename,
            "Rows": len(session["df"]),
            "Duration (s)": round(session["duration_sec"], 2),
            "Signal Columns": signal_text if signal_text else "None detected",
            "Marker Column": session["marker_column"] or "None detected",
            "Unique Markers": ", ".join(map(str, session["unique_markers"][:10])) if session["unique_markers"] else "None detected",
        }
    )

FILE_SUMMARY_DF = pd.DataFrame(summary_rows).sort_values("Filename").reset_index(drop=True)
display(FILE_SUMMARY_DF)


## Section 4 — Protocol and Marker Mapping

The mapping below assumes a common OpenBCI convention:

- `1 = LEFT start`
- `2 = LEFT stop`
- `3 = RIGHT start`
- `4 = RIGHT stop`

The dictionary is editable because LRJ and other guided sessions may need small adjustments. Marker transitions are converted into labeled intervals: **LEFT**, **RIGHT**, **REST**, or **UNKNOWN**.

If markers are missing or inconsistent, the notebook issues warnings and keeps going rather than crashing.


In [ ]:
MARKER_MAP = {
    "left_start": [1],
    "left_stop": [2],
    "right_start": [3],
    "right_stop": [4],
    "jaw_start": [],
    "jaw_stop": [],
}

FILE_PROTOCOL_OVERRIDES = {filename: session["protocol_tag"] for filename, session in SESSIONS.items()}
print("Edit FILE_PROTOCOL_OVERRIDES or MARKER_MAP if a session uses a different convention.")
display(pd.DataFrame({"Filename": list(FILE_PROTOCOL_OVERRIDES), "Protocol": list(FILE_PROTOCOL_OVERRIDES.values())}))


def normalize_marker_token(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text == "":
        return None
    try:
        numeric = float(text)
        if np.isfinite(numeric):
            if abs(numeric - round(numeric)) < 1e-9:
                return int(round(numeric))
            return numeric
    except ValueError:
        pass
    return text.upper()


def marker_matches(marker_value, allowed_values):
    for allowed in allowed_values:
        if isinstance(allowed, str):
            if str(marker_value).upper() == allowed.upper():
                return True
        else:
            try:
                if float(marker_value) == float(allowed):
                    return True
            except Exception:
                continue
    return False


def extract_marker_events(session):
    marker_column = session["marker_column"]
    if marker_column is None:
        return pd.DataFrame(columns=["time_sec", "marker"])

    raw_values = session["df"][marker_column].tolist()
    time_axis = session["time_axis_sec"]
    events = []
    previous = object()
    for time_sec, raw_value in zip(time_axis, raw_values):
        marker = normalize_marker_token(raw_value)
        if marker in [None, 0, 0.0, "0", "0.0"]:
            previous = marker
            continue
        if marker == previous:
            continue
        events.append({"time_sec": float(time_sec), "marker": marker})
        previous = marker
    return pd.DataFrame(events)


def build_labeled_intervals(session, marker_map):
    events_df = extract_marker_events(session)
    labeled_intervals = []
    active_label = None
    active_start = None

    for row in events_df.itertuples():
        marker = row.marker
        time_sec = float(row.time_sec)
        if marker_matches(marker, marker_map["left_start"]):
            active_label = "LEFT"
            active_start = time_sec
        elif marker_matches(marker, marker_map["left_stop"]):
            if active_label == "LEFT" and active_start is not None and time_sec > active_start:
                labeled_intervals.append({"start_sec": active_start, "end_sec": time_sec, "label": "LEFT"})
            active_label = None
            active_start = None
        elif marker_matches(marker, marker_map["right_start"]):
            active_label = "RIGHT"
            active_start = time_sec
        elif marker_matches(marker, marker_map["right_stop"]):
            if active_label == "RIGHT" and active_start is not None and time_sec > active_start:
                labeled_intervals.append({"start_sec": active_start, "end_sec": time_sec, "label": "RIGHT"})
            active_label = None
            active_start = None

    if not labeled_intervals:
        return pd.DataFrame(columns=["start_sec", "end_sec", "label", "duration_sec"])

    intervals_df = pd.DataFrame(labeled_intervals).sort_values("start_sec").reset_index(drop=True)
    rest_rows = []
    previous_end = 0.0
    for row in intervals_df.itertuples():
        if row.start_sec > previous_end:
            rest_rows.append({"start_sec": previous_end, "end_sec": row.start_sec, "label": "REST"})
        previous_end = max(previous_end, row.end_sec)
    session_end = float(session["time_axis_sec"][-1]) if len(session["time_axis_sec"]) else previous_end
    if session_end > previous_end:
        rest_rows.append({"start_sec": previous_end, "end_sec": session_end, "label": "REST"})

    if rest_rows:
        intervals_df = pd.concat([intervals_df, pd.DataFrame(rest_rows)], ignore_index=True)
        intervals_df = intervals_df.sort_values("start_sec").reset_index(drop=True)

    intervals_df["duration_sec"] = intervals_df["end_sec"] - intervals_df["start_sec"]
    intervals_df = intervals_df[intervals_df["duration_sec"] > 0].reset_index(drop=True)
    return intervals_df


INTERVAL_SUMMARY_ROWS = []
for filename, session in SESSIONS.items():
    marker_events = extract_marker_events(session)
    intervals_df = build_labeled_intervals(session, MARKER_MAP)
    session["marker_events"] = marker_events
    session["intervals"] = intervals_df

    if intervals_df.empty:
        warnings.warn(f"{filename}: no LEFT/RIGHT intervals were recovered from markers.")

    left_count = int((intervals_df["label"] == "LEFT").sum()) if not intervals_df.empty else 0
    right_count = int((intervals_df["label"] == "RIGHT").sum()) if not intervals_df.empty else 0
    rest_count = int((intervals_df["label"] == "REST").sum()) if not intervals_df.empty else 0

    INTERVAL_SUMMARY_ROWS.append(
        {
            "Filename": filename,
            "Protocol": FILE_PROTOCOL_OVERRIDES[filename],
            "Marker Events": len(marker_events),
            "LEFT Intervals": left_count,
            "RIGHT Intervals": right_count,
            "REST Intervals": rest_count,
        }
    )

INTERVAL_SUMMARY_DF = pd.DataFrame(INTERVAL_SUMMARY_ROWS).sort_values("Filename").reset_index(drop=True)
display(INTERVAL_SUMMARY_DF)


## Section 5 — Preprocessing

The EEG preprocessing stack uses:

- median/common reference subtraction
- 60 Hz notch filtering
- broad EEG bandpass (**1–40 Hz**)
- optional motor-band view (**8–30 Hz**)

The notebook will suggest potentially weak channels using simple variance/range checks, but it will **not** drop them silently. You can manually edit `BAD_CHANNELS` before preprocessing.


In [ ]:
def evaluate_channel_quality(session):
    signal_df = session["signal_df"].copy()
    rows = []
    for channel in signal_df.columns:
        values = clean_numeric(signal_df[channel]).dropna().to_numpy(dtype=float)
        if len(values) == 0:
            rows.append({"Channel": channel, "Variance": np.nan, "Range": np.nan, "Status": "FLATLINE"})
            continue
        variance = float(np.var(values))
        value_range = float(np.ptp(values))
        diff_std = float(np.std(np.diff(values))) if len(values) > 1 else 0.0
        dominant_fraction = float(pd.Series(np.round(values, 6)).value_counts(normalize=True).iloc[0])
        status = "GOOD"
        if variance < 1e-8 or value_range < 1e-4 or dominant_fraction > 0.98:
            status = "LOW_VARIANCE"
        elif diff_std > np.nanmedian(np.abs(values - np.nanmedian(values))) * 6:
            status = "NOISY"
        rows.append({
            "Channel": channel,
            "Variance": variance,
            "Range": value_range,
            "Diff Std": diff_std,
            "Dominant Fraction": dominant_fraction,
            "Status": status,
        })
    return pd.DataFrame(rows)


QUALITY_BY_FILE = {}
AUTO_BAD_CHANNEL_SUGGESTIONS = []
for filename, session in SESSIONS.items():
    quality_df = evaluate_channel_quality(session)
    QUALITY_BY_FILE[filename] = quality_df
    print(f"Channel quality — {filename}")
    display(quality_df)
    AUTO_BAD_CHANNEL_SUGGESTIONS.extend(quality_df.loc[quality_df["Status"] != "GOOD", "Channel"].tolist())

AUTO_BAD_CHANNEL_SUGGESTIONS = sorted(set(AUTO_BAD_CHANNEL_SUGGESTIONS))
print("Auto-suggested bad channels (review manually before removal):", AUTO_BAD_CHANNEL_SUGGESTIONS)

BAD_CHANNELS = []
print("Manual BAD_CHANNELS list initialized. Edit this list if you want to exclude suggested channels.")


In [ ]:
def safe_filtfilt(b, a, values):
    values = np.asarray(values, dtype=float)
    if len(values) < max(len(a), len(b)) * 3:
        return values.copy()
    finite_values = values.copy()
    if np.isnan(finite_values).any():
        fill_value = np.nanmedian(finite_values)
        if not np.isfinite(fill_value):
            fill_value = 0.0
        finite_values = np.nan_to_num(finite_values, nan=fill_value)
    return filtfilt(b, a, finite_values)


def common_reference(signal_df):
    row_reference = signal_df.median(axis=1)
    return signal_df.sub(row_reference, axis=0)


def notch_filter(values, fs_hz, notch_hz=60.0, quality_factor=30.0):
    nyquist = 0.5 * fs_hz
    if notch_hz >= nyquist:
        return np.asarray(values, dtype=float)
    b, a = iirnotch(notch_hz, quality_factor, fs_hz)
    return safe_filtfilt(b, a, values)


def bandpass_filter(values, fs_hz, low_hz=1.0, high_hz=40.0, order=4):
    nyquist = 0.5 * fs_hz
    high_hz = min(high_hz, nyquist * 0.95)
    if high_hz <= low_hz:
        return np.asarray(values, dtype=float)
    b, a = butter(order, [low_hz / nyquist, high_hz / nyquist], btype="band")
    return safe_filtfilt(b, a, values)


def preprocess_session(session, low_hz=1.0, high_hz=40.0, excluded_channels=None):
    excluded_channels = set(excluded_channels or [])
    keep_channels = [channel for channel in session["signal_columns"] if channel not in excluded_channels]
    if not keep_channels:
        warnings.warn(f"{session['filename']}: all channels were excluded; using all detected channels instead.")
        keep_channels = list(session["signal_columns"])

    signal_df = session["signal_df"].loc[:, keep_channels].copy().apply(clean_numeric)
    referenced = common_reference(signal_df)
    fs_hz = float(session["fs_estimate_hz"])

    filtered = pd.DataFrame(index=referenced.index)
    for channel in referenced.columns:
        values = referenced[channel].to_numpy(dtype=float)
        values = notch_filter(values, fs_hz)
        values = bandpass_filter(values, fs_hz, low_hz=low_hz, high_hz=high_hz)
        filtered[channel] = values
    return filtered


for session in SESSIONS.values():
    session["preprocessed_eeg"] = preprocess_session(session, low_hz=1.0, high_hz=40.0, excluded_channels=BAD_CHANNELS)
    session["preprocessed_motor"] = preprocess_session(session, low_hz=8.0, high_hz=30.0, excluded_channels=BAD_CHANNELS)

PLOT_FILENAME = next(iter(SESSIONS))
PLOT_SESSION = SESSIONS[PLOT_FILENAME]
PLOT_CHANNELS = PLOT_SESSION["preprocessed_eeg"].columns[: min(3, len(PLOT_SESSION["preprocessed_eeg"].columns))]

if len(PLOT_CHANNELS) > 0:
    time_axis = PLOT_SESSION["time_axis_sec"]
    zoom_mask = time_axis <= min(8.0, time_axis[-1])
    fig, axes = plt.subplots(len(PLOT_CHANNELS), 1, figsize=(14, 3 * len(PLOT_CHANNELS)), sharex=True)
    if len(PLOT_CHANNELS) == 1:
        axes = [axes]
    for ax, channel in zip(axes, PLOT_CHANNELS):
        raw_values = PLOT_SESSION["signal_df"][channel].to_numpy(dtype=float)
        filtered_values = PLOT_SESSION["preprocessed_eeg"][channel].to_numpy(dtype=float)
        ax.plot(time_axis[zoom_mask], raw_values[zoom_mask], label="Raw", linewidth=1.0, alpha=0.75)
        ax.plot(time_axis[zoom_mask], filtered_values[zoom_mask], label="EEG preprocessing (1–40 Hz)", linewidth=1.1)
        ax.set_title(f"{PLOT_FILENAME} — {channel}")
        ax.set_ylabel("Amplitude")
        ax.legend(loc="upper right")
    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()
else:
    print(f"{PLOT_FILENAME} does not have enough retained channels for a before/after plot.")


## Section 6 — Windowed LEFT vs RIGHT Benchmark

The windowed benchmark uses classic overlapping EEG windows inside labeled **LEFT** and **RIGHT** intervals only. It ignores REST for the pure direction benchmark.

Features include:

- mean
- variance
- RMS
- mean absolute value
- waveform length
- mu power (8–12 Hz)
- beta power (13–30 Hz)
- total motor-band power (8–30 Hz)
- simple adjacent-channel asymmetry terms based on motor-band power


In [ ]:
WINDOW_SECONDS = 1.0
WINDOW_OVERLAP = 0.5


def bandpower(values, fs_hz, low_hz, high_hz):
    values = np.asarray(values, dtype=float)
    if len(values) < 8:
        return 0.0
    nperseg = min(len(values), max(32, int(fs_hz)))
    freqs, power = welch(values, fs=fs_hz, nperseg=nperseg)
    mask = (freqs >= low_hz) & (freqs <= high_hz)
    if not np.any(mask):
        return 0.0
    return float(np.trapz(power[mask], freqs[mask]))


def compute_feature_row(window_matrix, fs_hz, channel_names):
    feature_row = {}
    motor_powers = []
    window_matrix = np.asarray(window_matrix, dtype=float)

    for channel_idx, channel_name in enumerate(channel_names):
        values = window_matrix[:, channel_idx]
        values = np.nan_to_num(values, nan=np.nanmedian(values) if np.isfinite(np.nanmedian(values)) else 0.0)
        centered = values - np.median(values)
        diffs = np.diff(centered)

        feature_row[f"{channel_name}__mean"] = float(np.mean(centered))
        feature_row[f"{channel_name}__variance"] = float(np.var(centered))
        feature_row[f"{channel_name}__rms"] = float(np.sqrt(np.mean(centered ** 2)))
        feature_row[f"{channel_name}__mav"] = float(np.mean(np.abs(centered)))
        feature_row[f"{channel_name}__waveform_length"] = float(np.sum(np.abs(diffs)))
        feature_row[f"{channel_name}__mu_power"] = bandpower(centered, fs_hz, 8.0, 12.0)
        feature_row[f"{channel_name}__beta_power"] = bandpower(centered, fs_hz, 13.0, 30.0)
        motor_power = bandpower(centered, fs_hz, 8.0, 30.0)
        feature_row[f"{channel_name}__motor_power"] = motor_power
        motor_powers.append(motor_power)

    for pair_idx in range(len(motor_powers) - 1):
        feature_row[f"pair_{pair_idx + 1}_{pair_idx + 2}__motor_diff"] = motor_powers[pair_idx] - motor_powers[pair_idx + 1]
    return feature_row


def build_windowed_feature_table(session, window_seconds=1.0, overlap_fraction=0.5):
    intervals_df = session["intervals"]
    if intervals_df.empty:
        return pd.DataFrame()

    signal_df = session["preprocessed_eeg"]
    time_axis = session["time_axis_sec"]
    fs_hz = float(session["fs_estimate_hz"])
    window_samples = max(8, int(round(window_seconds * fs_hz)))
    step_samples = max(1, int(round(window_samples * (1.0 - overlap_fraction))))

    rows = []
    for interval in intervals_df.itertuples():
        if interval.label not in CLASS_ORDER:
            continue
        eligible = np.where((time_axis >= interval.start_sec) & (time_axis <= interval.end_sec))[0]
        if len(eligible) < window_samples:
            continue
        for start_idx in range(int(eligible[0]), int(eligible[-1] - window_samples + 2), step_samples):
            end_idx = start_idx + window_samples
            if end_idx > len(signal_df):
                continue
            end_time = float(time_axis[end_idx - 1])
            if end_time > interval.end_sec:
                continue
            window_matrix = signal_df.iloc[start_idx:end_idx].to_numpy(dtype=float)
            features = compute_feature_row(window_matrix, fs_hz, signal_df.columns)
            features.update(
                {
                    "filename": session["filename"],
                    "protocol": session["protocol_tag"],
                    "feature_type": "windowed",
                    "start_time": float(time_axis[start_idx]),
                    "label": interval.label,
                }
            )
            rows.append(features)
    return pd.DataFrame(rows)


WINDOWED_TABLES = []
for session in SESSIONS.values():
    table = build_windowed_feature_table(session, window_seconds=WINDOW_SECONDS, overlap_fraction=WINDOW_OVERLAP)
    if not table.empty:
        WINDOWED_TABLES.append(table)

WINDOWED_FEATURE_DF = pd.concat(WINDOWED_TABLES, ignore_index=True) if WINDOWED_TABLES else pd.DataFrame()

print("Windowed class counts:")
if WINDOWED_FEATURE_DF.empty:
    print("No windowed LEFT/RIGHT samples were extracted.")
else:
    display(WINDOWED_FEATURE_DF["label"].value_counts().rename_axis("Label").reset_index(name="Count"))
    display(WINDOWED_FEATURE_DF.head())


## Section 7 — Event-Based LEFT vs RIGHT Benchmark

Windowed labels are coarse because long movement blocks contain mixtures of stronger and weaker neural activity. Event-based extraction is cleaner because it focuses on discrete LEFT and RIGHT starts.

This matters for the project: **LRJ is not a random one-off**. It is a structured hybrid event/count protocol family for **LEFT**, **RIGHT**, and **JAW**, and the GUI can continuously collect clean event-based data for all three movements.

The event benchmark below extracts a short post-onset window after each LEFT or RIGHT start marker, with optional pre-event baseline support.


In [ ]:
EVENT_WINDOW_START = 0.0
EVENT_WINDOW_END = 1.5
PRE_EVENT_BASELINE = (-0.5, 0.0)


def build_event_feature_table(session, event_window=(0.0, 1.5), baseline_window=(-0.5, 0.0)):
    marker_events = session["marker_events"]
    if marker_events.empty:
        return pd.DataFrame()

    signal_df = session["preprocessed_eeg"]
    time_axis = session["time_axis_sec"]
    fs_hz = float(session["fs_estimate_hz"])
    rows = []

    for row in marker_events.itertuples():
        label = None
        if marker_matches(row.marker, MARKER_MAP["left_start"]):
            label = "LEFT"
        elif marker_matches(row.marker, MARKER_MAP["right_start"]):
            label = "RIGHT"
        if label is None:
            continue

        event_time = float(row.time_sec)
        event_start = event_time + event_window[0]
        event_end = event_time + event_window[1]
        baseline_start = event_time + baseline_window[0]
        baseline_end = event_time + baseline_window[1]

        event_mask = (time_axis >= event_start) & (time_axis <= event_end)
        if event_mask.sum() < max(8, int(0.5 * fs_hz)):
            continue
        event_matrix = signal_df.loc[event_mask, :].to_numpy(dtype=float)

        baseline_mask = (time_axis >= baseline_start) & (time_axis <= baseline_end)
        if baseline_mask.sum() >= 4:
            baseline_matrix = signal_df.loc[baseline_mask, :].to_numpy(dtype=float)
            event_matrix = event_matrix - baseline_matrix.mean(axis=0, keepdims=True)

        features = compute_feature_row(event_matrix, fs_hz, signal_df.columns)
        features.update(
            {
                "filename": session["filename"],
                "protocol": session["protocol_tag"],
                "feature_type": "event_based",
                "event_time": event_time,
                "event_type": "start_marker",
                "label": label,
            }
        )
        rows.append(features)
    return pd.DataFrame(rows)


EVENT_TABLES = []
for session in SESSIONS.values():
    table = build_event_feature_table(session, event_window=(EVENT_WINDOW_START, EVENT_WINDOW_END), baseline_window=PRE_EVENT_BASELINE)
    if not table.empty:
        EVENT_TABLES.append(table)

EVENT_FEATURE_DF = pd.concat(EVENT_TABLES, ignore_index=True) if EVENT_TABLES else pd.DataFrame()

print("Event-based class counts:")
if EVENT_FEATURE_DF.empty:
    print("No event-based LEFT/RIGHT samples were extracted.")
else:
    display(EVENT_FEATURE_DF["label"].value_counts().rename_axis("Label").reset_index(name="Count"))
    display(EVENT_FEATURE_DF.head())


## Section 8 — Model Comparison

The notebook compares four classical models:

- Logistic Regression
- Linear Discriminant Analysis
- Random Forest
- SVM

Both feature families are evaluated:

- **windowed**
- **event-based**

If multiple recordings are available, the notebook prefers **cross-session held-out-file evaluation**. Otherwise it falls back to a within-session train/test split.


In [ ]:
MODEL_SPECS = {
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)),
        ]
    ),
    "Linear Discriminant Analysis": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LinearDiscriminantAnalysis()),
        ]
    ),
    "Random Forest": Pipeline(
        [
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    random_state=42,
                ),
            )
        ]
    ),
    "SVM": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)),
        ]
    ),
}


def prepare_feature_frame(feature_df):
    if feature_df.empty:
        return pd.DataFrame(), []
    feature_columns = [column for column in feature_df.columns if "__" in column]
    x = feature_df[feature_columns].fillna(0.0)
    return x, feature_columns


def compute_metric_bundle(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASS_ORDER,
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": float(np.mean(f1)),
        "left_precision": precision[0],
        "left_recall": recall[0],
        "right_precision": precision[1],
        "right_recall": recall[1],
        "left_f1": f1[0],
        "right_f1": f1[1],
        "left_support": support[0],
        "right_support": support[1],
    }


def predict_label_probabilities(fitted_model, x_frame):
    proba = fitted_model.predict_proba(x_frame)
    class_to_index = {label: idx for idx, label in enumerate(fitted_model.classes_)}
    left_prob = proba[:, class_to_index["LEFT"]]
    right_prob = proba[:, class_to_index["RIGHT"]]
    return left_prob, right_prob


def evaluate_within_session(feature_df, feature_columns, model_name, estimator):
    x = feature_df[feature_columns].fillna(0.0)
    y = feature_df["label"]
    meta_cols = [column for column in ["filename", "start_time", "event_time", "feature_type"] if column in feature_df.columns]
    meta = feature_df[meta_cols]

    train_x, test_x, train_y, test_y, train_meta, test_meta = train_test_split(
        x,
        y,
        meta,
        test_size=0.25,
        random_state=42,
        stratify=y,
    )
    fitted = clone(estimator)
    fitted.fit(train_x, train_y)
    pred_y = fitted.predict(test_x)
    left_prob, right_prob = predict_label_probabilities(fitted, test_x)
    metrics = compute_metric_bundle(test_y, pred_y)

    pred_df = test_meta.reset_index(drop=True).copy()
    pred_df["true_label"] = test_y.reset_index(drop=True)
    pred_df["pred_label"] = pd.Series(pred_y)
    pred_df["p_left"] = pd.Series(left_prob)
    pred_df["p_right"] = pd.Series(right_prob)

    return {
        "model": model_name,
        "evaluation_type": "within_session",
        "metrics": metrics,
        "predictions": pred_df,
        "fold_summary": pd.DataFrame([{"Fold": "random_split", "Test File": "mixed", "Samples": len(pred_df), "Macro F1": metrics["macro_f1"]}]),
    }


def evaluate_cross_session(feature_df, feature_columns, model_name, estimator):
    unique_files = sorted(feature_df["filename"].unique())
    if len(unique_files) < 2:
        return None

    fold_rows = []
    prediction_rows = []
    for test_file in unique_files:
        train_df = feature_df[feature_df["filename"] != test_file].reset_index(drop=True)
        test_df = feature_df[feature_df["filename"] == test_file].reset_index(drop=True)
        if train_df.empty or test_df.empty:
            continue
        if set(train_df["label"]) != set(CLASS_ORDER):
            continue

        fitted = clone(estimator)
        fitted.fit(train_df[feature_columns].fillna(0.0), train_df["label"])
        pred_y = fitted.predict(test_df[feature_columns].fillna(0.0))
        left_prob, right_prob = predict_label_probabilities(fitted, test_df[feature_columns].fillna(0.0))
        fold_metrics = compute_metric_bundle(test_df["label"], pred_y)

        fold_rows.append({"Fold": f"holdout_{test_file}", "Test File": test_file, "Samples": len(test_df), "Macro F1": fold_metrics["macro_f1"]})

        pred_df = test_df[[column for column in ["filename", "start_time", "event_time", "feature_type"] if column in test_df.columns]].copy()
        pred_df["true_label"] = test_df["label"].values
        pred_df["pred_label"] = pred_y
        pred_df["p_left"] = left_prob
        pred_df["p_right"] = right_prob
        prediction_rows.append(pred_df)

    if not prediction_rows:
        return None

    all_predictions = pd.concat(prediction_rows, ignore_index=True)
    metrics = compute_metric_bundle(all_predictions["true_label"], all_predictions["pred_label"])
    return {
        "model": model_name,
        "evaluation_type": "cross_session",
        "metrics": metrics,
        "predictions": all_predictions,
        "fold_summary": pd.DataFrame(fold_rows),
    }


def run_feature_benchmark(feature_df, feature_type):
    if feature_df.empty:
        return []
    if set(feature_df["label"]) != set(CLASS_ORDER):
        warnings.warn(f"{feature_type}: both LEFT and RIGHT labels were not available. Skipping benchmark.")
        return []

    x, feature_columns = prepare_feature_frame(feature_df)
    if x.empty:
        return []

    results = []
    for model_name, estimator in MODEL_SPECS.items():
        result = evaluate_cross_session(feature_df, feature_columns, model_name, estimator)
        if result is None:
            result = evaluate_within_session(feature_df, feature_columns, model_name, estimator)
        result["feature_type"] = feature_type
        result["feature_columns"] = feature_columns
        results.append(result)
    return results


WINDOWED_RESULTS = run_feature_benchmark(WINDOWED_FEATURE_DF, "windowed")
EVENT_RESULTS = run_feature_benchmark(EVENT_FEATURE_DF, "event_based")
ALL_RESULTS = WINDOWED_RESULTS + EVENT_RESULTS

if not ALL_RESULTS:
    raise RuntimeError("No benchmark results were produced. Check the uploaded files and marker mapping.")

RESULTS_SUMMARY_DF = pd.DataFrame(
    [
        {
            "Model": result["model"],
            "Feature Type": result["feature_type"],
            "Evaluation Type": result["evaluation_type"],
            "Accuracy": result["metrics"]["accuracy"],
            "Macro F1": result["metrics"]["macro_f1"],
            "LEFT Precision": result["metrics"]["left_precision"],
            "LEFT Recall": result["metrics"]["left_recall"],
            "RIGHT Precision": result["metrics"]["right_precision"],
            "RIGHT Recall": result["metrics"]["right_recall"],
        }
        for result in ALL_RESULTS
    ]
).sort_values(["Macro F1", "Accuracy"], ascending=False).reset_index(drop=True)

display(RESULTS_SUMMARY_DF)

BEST_BY_FEATURE = {}
for feature_type in ["windowed", "event_based"]:
    feature_results = [result for result in ALL_RESULTS if result["feature_type"] == feature_type]
    if feature_results:
        BEST_BY_FEATURE[feature_type] = max(feature_results, key=lambda result: result["metrics"]["macro_f1"])

BEST_OVERALL = max(ALL_RESULTS, key=lambda result: result["metrics"]["macro_f1"])
FINAL_MODEL = clone(MODEL_SPECS[BEST_OVERALL["model"]]).fit(
    (WINDOWED_FEATURE_DF if BEST_OVERALL["feature_type"] == "windowed" else EVENT_FEATURE_DF)[BEST_OVERALL["feature_columns"]].fillna(0.0),
    (WINDOWED_FEATURE_DF if BEST_OVERALL["feature_type"] == "windowed" else EVENT_FEATURE_DF)["label"],
)

print(f"Best overall benchmark: {BEST_OVERALL['model']} on {BEST_OVERALL['feature_type']} ({BEST_OVERALL['evaluation_type']})")


## Section 9 — Results

The tables below summarize the main benchmark results. Detailed confusion matrices and classification reports are shown for the best **windowed** model and the best **event-based** model.


In [ ]:
for feature_type in ["windowed", "event_based"]:
    if feature_type not in BEST_BY_FEATURE:
        print(f"No {feature_type} benchmark result is available.")
        continue

    result = BEST_BY_FEATURE[feature_type]
    print(f"Best {feature_type} result: {result['model']} ({result['evaluation_type']})")
    display(result["fold_summary"])

    report_df = pd.DataFrame(
        classification_report(
            result["predictions"]["true_label"],
            result["predictions"]["pred_label"],
            labels=CLASS_ORDER,
            output_dict=True,
            zero_division=0,
        )
    ).T
    display(report_df)

    cm = confusion_matrix(result["predictions"]["true_label"], result["predictions"]["pred_label"], labels=CLASS_ORDER)
    fig, ax = plt.subplots(figsize=(5, 5))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_ORDER).plot(ax=ax, colorbar=False)
    ax.set_title(f"Confusion Matrix — {feature_type} / {result['model']}")
    plt.show()


## Section 10 — Probability / Confidence Timeline

The timeline below runs the best available model on one selected file and shows predicted **LEFT** and **RIGHT** confidence over time, with true marker intervals overlaid when available.

If the best model is event-based, the plot will show event-centered probabilities rather than a fully continuous trace.


In [ ]:
TIMELINE_FILENAME = next(iter(SESSIONS))
TIMELINE_SESSION = SESSIONS[TIMELINE_FILENAME]
TIMELINE_RESULT = BEST_OVERALL
TIMELINE_FEATURE_COLUMNS = TIMELINE_RESULT["feature_columns"]

if TIMELINE_RESULT["feature_type"] == "windowed":
    timeline_df = build_windowed_feature_table(TIMELINE_SESSION, window_seconds=WINDOW_SECONDS, overlap_fraction=WINDOW_OVERLAP)
    time_column = "start_time"
else:
    timeline_df = build_event_feature_table(TIMELINE_SESSION, event_window=(EVENT_WINDOW_START, EVENT_WINDOW_END), baseline_window=PRE_EVENT_BASELINE)
    time_column = "event_time"

if timeline_df.empty:
    print(f"No timeline samples were available for {TIMELINE_FILENAME} using the best feature type.")
else:
    x_timeline = timeline_df.reindex(columns=TIMELINE_FEATURE_COLUMNS, fill_value=0.0)
    left_prob, right_prob = predict_label_probabilities(FINAL_MODEL, x_timeline)
    timeline_df = timeline_df.copy()
    timeline_df["p_left"] = left_prob
    timeline_df["p_right"] = right_prob

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(timeline_df[time_column], timeline_df["p_left"], label="P(LEFT)", linewidth=1.5, color="#2563eb")
    ax.plot(timeline_df[time_column], timeline_df["p_right"], label="P(RIGHT)", linewidth=1.5, color="#dc2626")

    intervals_df = TIMELINE_SESSION["intervals"]
    if not intervals_df.empty:
        for row in intervals_df.itertuples():
            if row.label == "LEFT":
                ax.axvspan(row.start_sec, row.end_sec, color="#bfdbfe", alpha=0.25)
            elif row.label == "RIGHT":
                ax.axvspan(row.start_sec, row.end_sec, color="#fecaca", alpha=0.22)

    ax.set_title(f"Probability Timeline — {TIMELINE_FILENAME} ({TIMELINE_RESULT['feature_type']})")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Probability")
    ax.set_ylim(-0.02, 1.02)
    ax.legend(loc="upper right")
    plt.show()


## Section 11 — Interpretation

The notebook should end by being clear about what worked and what did not. That includes whether event-based extraction improved label quality, why left/right EEG remains difficult, and why LRJ/guided event sessions matter for future training.


In [ ]:
def safe_metric(result_dict, feature_type, key):
    if feature_type not in result_dict:
        return None
    return result_dict[feature_type]["metrics"].get(key)

window_macro_f1 = safe_metric(BEST_BY_FEATURE, "windowed", "macro_f1")
event_macro_f1 = safe_metric(BEST_BY_FEATURE, "event_based", "macro_f1")

comparison_line = "Event-based extraction did not produce a valid comparison in this upload set."
if window_macro_f1 is not None and event_macro_f1 is not None:
    if event_macro_f1 > window_macro_f1:
        comparison_line = f"Event-based extraction improved macro F1 from {window_macro_f1:.3f} to {event_macro_f1:.3f}, which supports the idea that cleaner onset-centered data is more informative than coarse block labeling."
    elif event_macro_f1 < window_macro_f1:
        comparison_line = f"Windowed extraction outperformed the event-based benchmark in this upload set ({window_macro_f1:.3f} vs {event_macro_f1:.3f}), suggesting that the current event labels or event windows may still need refinement."
    else:
        comparison_line = f"Windowed and event-based benchmarks were similar in this upload set (both about {window_macro_f1:.3f} macro F1)."

interpretation = f"""
### Interpretation

**Is LEFT vs RIGHT decoding strong or weak?**  
The EEG directional benchmark remains the weaker branch of the project. Even when usable patterns appear, performance is more sensitive to session variation and label quality than the jaw decoder.

**What did event-based extraction add?**  
{comparison_line}

**Why is EEG direction harder than jaw click detection?**  
Directional EEG intent has lower signal-to-noise ratio, fewer strong transient signatures, and more session drift. Jaw clench produces a larger, more repeatable biosignal.

**Why use a hybrid design instead of EEG only?**  
Because an EEG-only interface would be less reliable for discrete control. The hybrid design lets jaw handle the stronger click-like action while EEG remains a directional research branch.

**Why are LRJ/guided event sessions valuable?**  
LRJ sessions are a structured hybrid event/count protocol family, not random one-offs. They provide cleaner event-centered LEFT, RIGHT, and JAW data and are valuable future training data for improving event-based decoding.
"""
display(Markdown(interpretation))


## Section 12 — Connection to Final Project

The final summary ties the benchmark back to the actual hybrid BCI system.


In [ ]:
final_summary = f"""
### Final Project Connection

- **Jaw decoding** provides the strongest discrete click control.
- **EEG LEFT/RIGHT** remains a useful directional research branch, but it is weaker and less stable than jaw.
- The final system combines the best available pieces:
  - **jaw = reliable click**
  - **EEG = directional intent exploration**
  - **calibration/adaptation = session-specific stabilization**
  - **GUI/game = real-time demonstration layer**

The benchmark in this notebook is intentionally honest: it shows effort, model comparison, and the reasons EEG-only control is not enough yet.
"""
display(Markdown(final_summary))
